# Lab 9: Transfer Learning with BERT

In this lab we are going to be looking at fine-tuning a BERT model to carry out a sequence classification task.

We are going to load in some text from a small number of books in the `Gutenberg corpus` and see if we can train a classifier to classify which book a piece of text is from 

First lets pull up all of the filenames available.

In [1]:
import os,random,math
TRAINING_DIR="../02_ngram_language_models/sentence-completion/Holmes_Training_Data" #this needs to be the parent directory for the training corpus

filenames=os.listdir(TRAINING_DIR)
n=len(filenames)
print("There are {} files in the training directory: {}".format(n,TRAINING_DIR))

#print(filenames)

There are 522 files in the training directory: ../02_ngram_language_models/sentence-completion/Holmes_Training_Data


In [2]:
filepath=os.path.join(TRAINING_DIR,"EMMA10.TXT")
filepath

'../02_ngram_language_models/sentence-completion/Holmes_Training_Data\\EMMA10.TXT'

We are going to create a Book class to store the text from a class and do some very basic pre-processing.  We need to 
* load in the text line-by-line
* get rid of header lines (the stuff in the file before the line which starts \*END\*THE SMALL PRINT!)
* make chunks of text which are longer than 1 line.  These should be easier to classify.  We will try to get sentences - but some chunks may contain multiple sentences.  We are not going to worry about this here.
* return some labelled data split randomly between training and testing

In [3]:
class Book():
    TRAINING_DIR="../02_ngram_language_models/sentence-completion/Holmes_Training_Data"
    header_end="*END*THE SMALL PRINT!"
    seed=53
   
    def __init__(self,filename,label=""):
        self.filename=filename
        self.loadfile()
        self.make_chunks()
        if label=="":
            self.label=self.filename
        else:
            self.label=label
        
    def loadfile(self):
        filepath=os.path.join(Book.TRAINING_DIR,self.filename)
        self.lines=[]
        beyond_header=False
        try:
            with open(filepath) as instream:
                for line in instream:
                    line=line.rstrip()
                    
                    if len(line)>0 and beyond_header:
                        self.lines.append(line)
                    if line.startswith(Book.header_end):
                        beyond_header=True
        except UnicodeDecodeError:
            print(f"UnicodeDecodeError processing {filepath}")
    
    def length(self):
        return len(self.chunks)
    
    def head(self,n=10):
        return self.chunks[:n]
    
    def make_chunks(self):
        self.chunks=[]
        
        current=""
        for line in self.lines:
            current+=line+" "
            if line.endswith("."):
                self.chunks.append(current.rstrip())
                current=""
            
    def get_labelled_data(self,split=0.8):
        labelled_data=[(chunk,self.label) for chunk in self.chunks]
        random.seed(Book.seed)
        random.shuffle(labelled_data)
        index=int(self.length()*split)
        test_index = int((self.length()-index) / 2)
        return (labelled_data[:index], labelled_data[index: -test_index], labelled_data[-test_index:])
                

## Exercise 1
- Create an instance of a Book() and store it in the variable `emma`.  The filename is `EMMA10.TXT` and the label should be `Emma`
- Check the number of sentences = 2028 and have a look at the first 10 sentence
- Repeat to create `ivanhoe` to store the text from `IVNHO12.TXT` with the label `Ivanhoe`.  
- The number of sentences in `ivanhoe` should be 1743

In [4]:
emma = Book("EMMA10.TXT", "Emma")

In [5]:
emma.length()

2028

In [6]:
emma.head()

['VOLUME I CHAPTER I Emma Woodhouse, handsome, clever, and rich, with a comfortable home and happy disposition, seemed to unite some of the best blessings of existence; and had lived nearly twenty-one years in the world with very little to distress or vex her.',
 "She was the youngest of the two daughters of a most affectionate, indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period.  Her mother had died too long ago for her to have more than an indistinct remembrance of her caresses; and her place had been supplied by an excellent woman as governess, who had fallen little short of a mother in affection.",
 "Sixteen years had Miss Taylor been in Mr. Woodhouse's family, less as a governess than a friend, very fond of both daughters, but particularly of Emma.  Between them it was more the intimacy of sisters.  Even before Miss Taylor had ceased to hold the nominal office of governess, the mildness of her temper had hardly a

In [7]:
ivanhoe = Book("IVNHO12.TXT", "Ivanhoe")

In [8]:
ivanhoe.length()

1743

In [9]:
ivanhoe.head()

['IVANHOE.',
 "CHAPTER I Thus communed these; while to their lowly dome, The full-fed swine return'd with evening home; Compell'd, reluctant, to the several sties, With din obstreperous, and ungrateful cries.",
 "               Pope's _Odyssey_.",
 'In that pleasant district of merry England which is watered by the river Don, there extended in ancient times a large forest, covering the greater part of the beautiful hills and valleys which lie between Sheffield and the pleasant town of Doncaster.  The remains of this extensive wood are still to be seen at the noble seats of Wentworth, of Warncliffe Park, and around Rotherham.  Here haunted of yore the fabulous Dragon of Wantley; here were fought many of the most desperate battles during the Civil Wars of the Roses; and here also flourished in ancient times those bands of gallant outlaws, whose deeds have been rendered so popular in English song.',
 'Such being our chief scene, the date of our story refers to a period towards the end of 

## Exercise 2
- Use the `get_labelled_data()` method to get a training and testing portion from each book (split = 80%).  
- Create 2 pandas dataframes
    - 1 dataframe called `training_df` with all of the training data (from both books)
    - 1 dataframe called  `testing_df`  with all of the test data (from both books).  
    - The columns of both dataframes should be `text` and `label`

In [10]:
train_emma, valid_emma, test_emma = emma.get_labelled_data()
train_ivanhoe, valid_ivanhoe, test_ivanhoe = ivanhoe.get_labelled_data()

In [11]:
print(*train_emma[:2], sep="\n")
print("-" * 50)
print(*valid_emma[:2], sep="\n")
print("-" * 50)
print(*test_emma[:2], sep="\n")
print("-" * 50)
print(*train_ivanhoe[:2], sep="\n")
print("-" * 50)
print(*valid_ivanhoe[:2], sep="\n")
print("-" * 50)
print(*test_ivanhoe[:2], sep="\n")

('I made the match, you know, four years ago; and to have it take place, and be proved in the right, when so many people said Mr. Weston would never marry again, may comfort me for any thing." Mr. Knightley shook his head at her.  Her father fondly replied, "Ah! my dear, I wish you would not make matches and foretell things, for whatever you say always comes to pass.  Pray do not make any more matches." "I promise you to make none for myself, papa; but I must, indeed, for other people.  It is the greatest amusement in the world! And after such success, you know!--Every body said that Mr. Weston would never marry again.  Oh dear, no! Mr. Weston, who had been a widower so long, and who seemed so perfectly comfortable without a wife, so constantly occupied either in his business in town or among his friends here, always acceptable wherever he went, always cheerful-- Mr. Weston need not spend a single evening in the year alone if he did not like it.  Oh no! Mr. Weston certainly would never

In [12]:
print(f"Emma Sets legnth: Train set: {len(train_emma)}, Valid set: {len(valid_emma)}, Test Set: {len(test_emma)}")
print(f"Ivanhoe Sets legnth: Train set: {len(train_ivanhoe)}, Valid set: {len(valid_ivanhoe)}, Test Set: {len(test_ivanhoe)}")

Emma Sets legnth: Train set: 1622, Valid set: 203, Test Set: 203
Ivanhoe Sets legnth: Train set: 1394, Valid set: 175, Test Set: 174


In [114]:
import pandas as pd


training_df = pd.DataFrame((train_emma + train_ivanhoe), columns=["text", "label"])
valid_df = pd.DataFrame((valid_emma + valid_ivanhoe), columns=["text", "label"])
testing_df = pd.DataFrame((test_emma + test_ivanhoe), columns=["text", "label"])

In [14]:
training_df.head()

,text,label
0,"I made the match, you know, four years ago; an...",Emma
1,"She went, however; and when they reached the f...",Emma
2,"That it was time for every body to go, conclud...",Emma
3,He argued like a young man very much bent on d...,Emma
4,It was the arrival of this very invitation whi...,Emma


In [55]:
training_df.tail()

,text,label
3011,"``That large decayed oak,'' he said, ``marks t...",Ivanhoe
3012,"When the first moments of surprise were over, ...",Ivanhoe
3013,Lucas Beaumanoir hath settled that the death o...,Ivanhoe
3014,"``And if he is here,'' said Rowena, compelling...",Ivanhoe
3015,"In the midst of Prince John's cavalcade, he su...",Ivanhoe


In [15]:
valid_df.head()

,text,label
0,She had never been admitted before to be serio...,Emma
1,Emma remained very well pleased with this begi...,Emma
2,"She saw that Enscombe could not satisfy, and t...",Emma
3,Nothing wanting. Could not have imagined it.-...,Emma
4,"""We were too magnificent,"" said he. ""We allow...",Emma


In [56]:
valid_df.tail()

,text,label
373,"It is a single round tower, the wall curving i...",Ivanhoe
374,"``Brother Ben Samuel,'' said Isaac, ``my soul ...",Ivanhoe
375,"``What if six,'' continued Wamba, ``and we as ...",Ivanhoe
376,_Richard III._ When the Bl...,Ivanhoe
377,"De Bracy, being attached to the Templars, woul...",Ivanhoe


In [16]:
testing_df["label"].unique()

<ArrowStringArray>
['Emma', 'Ivanhoe']
Length: 2, dtype: str

In [17]:
testing_df.head()

,text,label
0,"I think him a very handsome young man, and his...",Emma
1,Mr. Knightley and Harriet!--It was an odd tete...,Emma
2,"Captain Weston, who had been considered, espec...",Emma
3,I hope I have a better taste than to think of ...,Emma
4,She laughed because she was disappointed; and ...,Emma


In [57]:
testing_df.tail()

,text,label
372,"And there was such a solemn melody, 'Twixt dol...",Ivanhoe
373,"``The castle burns,'' said Rebecca; ``it burns...",Ivanhoe
374,"and for the evil which thou hast sustained, se...",Ivanhoe
375,But although the lists rang with the applauses...,Ivanhoe
376,"``Such,'' he said, ``and so great should indee...",Ivanhoe


In [18]:
print(training_df.iloc[0, 0])
print(valid_df.iloc[0, 0])
print(testing_df.iloc[0, 0])

I made the match, you know, four years ago; and to have it take place, and be proved in the right, when so many people said Mr. Weston would never marry again, may comfort me for any thing." Mr. Knightley shook his head at her.  Her father fondly replied, "Ah! my dear, I wish you would not make matches and foretell things, for whatever you say always comes to pass.  Pray do not make any more matches." "I promise you to make none for myself, papa; but I must, indeed, for other people.  It is the greatest amusement in the world! And after such success, you know!--Every body said that Mr. Weston would never marry again.  Oh dear, no! Mr. Weston, who had been a widower so long, and who seemed so perfectly comfortable without a wife, so constantly occupied either in his business in town or among his friends here, always acceptable wherever he went, always cheerful-- Mr. Weston need not spend a single evening in the year alone if he did not like it.  Oh no! Mr. Weston certainly would never m

## Finetuning BERT to tell the difference between sentences from each book

Now we are going to look at building a classifier on top of BERT.  
The first thing we need to do is:
map the informative label names we have (`Emma` and `Ivanhoe`) to integers which will be used by BERT. 
In this simple case, we could just create a dictionary manually.  

However, the code below will make a sorted list (without duplicates) of all of the labelnames in the two dataframes 
and then generate a dictionary which maps each label name to an integer.


In [19]:
#first we need a map for the labels

#make a list of all of the unique labels in the training and testing dataframes
#sorting it means that it will also be in the same order (alphabetical) rather than depending on order in the training / testing data
# union() combines two sets and keeps only unique values.
labellist=sorted(list(set(training_df['label'].unique()).union(set(valid_df['label'].unique())).union(set(testing_df['label'].unique())))) 
print(labellist)
labels={label:i for i,label in enumerate(labellist)}
labels

['Emma', 'Ivanhoe']


{'Emma': 0, 'Ivanhoe': 1}

### Exercise 3
Write some code to create a reverse index for the labels which maps the numbers back to the more informative strings

This should result in something which looks as follows:

```
reverse_index={0:'Emma',1:'Ivanhoe'}
```

But obviously, you should create it automatically from the labels dictionary rather than typing it in!

In [20]:
reverse_index = {value : key for key, value in labels.items()}
reverse_index 

{0: 'Emma', 1: 'Ivanhoe'}

Now we need to store the data in a Dataset class.  
This inherits properties from torch's Dataset class and is what is expected.  

It is initialised by giving it dataframe from which it can extract a list of labels and a list of texts.   

It handles preprocessing including 

- adding CLS and SEP tokens at the beginning and end, 
- tokenization, 
- lower-casing truncation and padding.  

It also provides a `\_\_getitem\_\_()` method which allows the Dataset to be indexed into like a list i.e., `myDataset[3]` will return a pair which is the label and text with index 3.

In [21]:
import torch
import numpy as np
from transformers import BertTokenizer
# from torch.utils.data import Dataset, DataLoader

tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

class Dataset(torch.utils.data.Dataset):
    
    def __init__(self,df,column='text'):
        self.labels=[labels[label] for label in df['label']]
        self.texts=[tokenizer(text.lower(),padding='max_length',max_length=512,truncation=True,return_tensors="pt") for text in df[column]]
        
    def classes(self):
        return self.labels
    
    def __len__(self):
        return len(self.labels)
    
    def get_batch_labels(self,idx):
        return np.array(self.labels[idx])
    
    def get_batch_texts(self,idx):
        return self.texts[idx]
    
    def __getitem__(self,idx):
        batch_texts=self.get_batch_texts(idx)
        batch_y=self.get_batch_labels(idx)
        
        return batch_texts,batch_y
    

train_data=Dataset(training_df)
valid_data=Dataset(valid_df)
test_data=Dataset(testing_df)

Lets have a look at one of the dataset items.

The \_\_getitem\_\_ method allows us to index into the dataset and get a particular item

In [22]:
train_data[0] # (text, label)

({'input_ids': tensor([[  101,  1045,  2081,  1996,  2674,  1010,  2017,  2113,  1010,  2176,
           2086,  3283,  1025,  1998,  2000,  2031,  2009,  2202,  2173,  1010,
           1998,  2022,  4928,  1999,  1996,  2157,  1010,  2043,  2061,  2116,
           2111,  2056,  2720,  1012, 12755,  2052,  2196,  5914,  2153,  1010,
           2089,  7216,  2033,  2005,  2151,  2518,  1012,  1000,  2720,  1012,
           5000,  3051,  3184,  2010,  2132,  2012,  2014,  1012,  2014,  2269,
          13545,  2135,  3880,  1010,  1000,  6289,   999,  2026,  6203,  1010,
           1045,  4299,  2017,  2052,  2025,  2191,  3503,  1998, 18921, 23567,
           2477,  1010,  2005,  3649,  2017,  2360,  2467,  3310,  2000,  3413,
           1012, 11839,  2079,  2025,  2191,  2151,  2062,  3503,  1012,  1000,
           1000,  1045,  4872,  2017,  2000,  2191,  3904,  2005,  2870,  1010,
          13008,  1025,  2021,  1045,  2442,  1010,  5262,  1010,  2005,  2060,
           2111,  1012,  2

In [23]:
print(len(train_data[0]))

2


In [24]:
train_data[0][1]

array(0)

In [25]:
input_ex = train_data[0][0]
print(len(input_ex))
print(type(input_ex))


3
<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [26]:
print(list(input_ex.keys()))

['input_ids', 'token_type_ids', 'attention_mask']


In [27]:
# Token IDs after tokenization.
print(input_ex.input_ids)
print(input_ex.input_ids.shape) # ( batchsize= one input,max_length=512)

tensor([[  101,  1045,  2081,  1996,  2674,  1010,  2017,  2113,  1010,  2176,
          2086,  3283,  1025,  1998,  2000,  2031,  2009,  2202,  2173,  1010,
          1998,  2022,  4928,  1999,  1996,  2157,  1010,  2043,  2061,  2116,
          2111,  2056,  2720,  1012, 12755,  2052,  2196,  5914,  2153,  1010,
          2089,  7216,  2033,  2005,  2151,  2518,  1012,  1000,  2720,  1012,
          5000,  3051,  3184,  2010,  2132,  2012,  2014,  1012,  2014,  2269,
         13545,  2135,  3880,  1010,  1000,  6289,   999,  2026,  6203,  1010,
          1045,  4299,  2017,  2052,  2025,  2191,  3503,  1998, 18921, 23567,
          2477,  1010,  2005,  3649,  2017,  2360,  2467,  3310,  2000,  3413,
          1012, 11839,  2079,  2025,  2191,  2151,  2062,  3503,  1012,  1000,
          1000,  1045,  4872,  2017,  2000,  2191,  3904,  2005,  2870,  1010,
         13008,  1025,  2021,  1045,  2442,  1010,  5262,  1010,  2005,  2060,
          2111,  1012,  2009,  2003,  1996,  4602,  

In [28]:
# Used for sentence-pair tasks.
#Important in: Question Answering, Next Sentence Prediction, Sentence pair classification
# Less important for single-text classification.
print(input_ex.token_type_ids)
print(input_ex.token_type_ids.shape)

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0

In [29]:
# Tells BERT which tokens are real and which are padding.
print(input_ex.attention_mask)
print(input_ex.attention_mask.shape)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0

We can also just look at the text (or label) for an item as follows.  Note that the tokens have been replaced by their indices in the BERT wordpiece vocabulary.

In [30]:
train_data.texts[0]

{'input_ids': tensor([[  101,  1045,  2081,  1996,  2674,  1010,  2017,  2113,  1010,  2176,
          2086,  3283,  1025,  1998,  2000,  2031,  2009,  2202,  2173,  1010,
          1998,  2022,  4928,  1999,  1996,  2157,  1010,  2043,  2061,  2116,
          2111,  2056,  2720,  1012, 12755,  2052,  2196,  5914,  2153,  1010,
          2089,  7216,  2033,  2005,  2151,  2518,  1012,  1000,  2720,  1012,
          5000,  3051,  3184,  2010,  2132,  2012,  2014,  1012,  2014,  2269,
         13545,  2135,  3880,  1010,  1000,  6289,   999,  2026,  6203,  1010,
          1045,  4299,  2017,  2052,  2025,  2191,  3503,  1998, 18921, 23567,
          2477,  1010,  2005,  3649,  2017,  2360,  2467,  3310,  2000,  3413,
          1012, 11839,  2079,  2025,  2191,  2151,  2062,  3503,  1012,  1000,
          1000,  1045,  4872,  2017,  2000,  2191,  3904,  2005,  2870,  1010,
         13008,  1025,  2021,  1045,  2442,  1010,  5262,  1010,  2005,  2060,
          2111,  1012,  2009,  2003,  

In [31]:
list(train_data.texts[0].keys())

['input_ids', 'token_type_ids', 'attention_mask']

### Exercise 4
Can you turn the token ids back into subwords for one of the inputs?

In [32]:
text5 = train_data.texts[5]
text5['input_ids']

tensor([[  101,  2002,  2234,  2176,  2335,  1037,  2154,  2005,  1037,  2733,
          1012,  2002,  2056,  1010,  2013,  1996,  2034,  1010,  2009,  2001,
          1037,  2200,  2204,  4066,  1011,  1011,  2029,  2001,  2256,  2307,
          7216,  1025,  2021,  1996,  2033,  3022,  4244,  2024,  1037, 21794,
         12087,  1012,  1045,  3246,  7188,  3532, 10323,  1005,  1055,  2210,
          3924,  2031,  1996,  2033,  3022,  4244,  1010,  2016,  2097,  4604,
          2005,  6890,  1012,  1000,  1000,  2026,  2269,  1998,  3680,  1012,
         12755,  2024,  2012,  1996,  4410,  2012,  2023,  2617,  1010,  1000,
          2056,  3581, 10888,  1010,  1000, 12843,  1996,  9859,  1997,  1996,
          2160,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

- outer [   ]  = batch dimension
- inner [   ]  = token sequence

BAtch size equal len(tensor) in this case len(text5['input_ids'])

One batch example:

`tensor([[101, 1045, 2293, 102, ...]])`

Two batches example:

`
tensor([
    [101, 1045, 2293, 102, 0, 0],
    [101, 2017, 2293, 2129, 102, 0]
])
`

In [33]:
text5['input_ids'].shape

torch.Size([1, 512])

In [34]:
len(text5['input_ids'])

1

In [35]:
# convert_ids_to_tokens() expects [List[int]] such as [101, 1045, 2293, 102]
# NOT [[101, 1045, 2293, 102, ...]]
tokenizer.convert_ids_to_tokens(text5['input_ids'][0])

['[CLS]',
 'he',
 'came',
 'four',
 'times',
 'a',
 'day',
 'for',
 'a',
 'week',
 '.',
 'he',
 'said',
 ',',
 'from',
 'the',
 'first',
 ',',
 'it',
 'was',
 'a',
 'very',
 'good',
 'sort',
 '-',
 '-',
 'which',
 'was',
 'our',
 'great',
 'comfort',
 ';',
 'but',
 'the',
 'me',
 '##as',
 '##les',
 'are',
 'a',
 'dreadful',
 'complaint',
 '.',
 'i',
 'hope',
 'whenever',
 'poor',
 'isabella',
 "'",
 's',
 'little',
 'ones',
 'have',
 'the',
 'me',
 '##as',
 '##les',
 ',',
 'she',
 'will',
 'send',
 'for',
 'perry',
 '.',
 '"',
 '"',
 'my',
 'father',
 'and',
 'mrs',
 '.',
 'weston',
 'are',
 'at',
 'the',
 'crown',
 'at',
 'this',
 'moment',
 ',',
 '"',
 'said',
 'frank',
 'churchill',
 ',',
 '"',
 'examining',
 'the',
 'capabilities',
 'of',
 'the',
 'house',
 '.',
 '[SEP]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[

In [36]:
training_df.iloc[5, 0]

'He came four times a day for a week.  He said, from the first, it was a very good sort--which was our great comfort; but the measles are a dreadful complaint.  I hope whenever poor Isabella\'s little ones have the measles, she will send for Perry." "My father and Mrs. Weston are at the Crown at this moment," said Frank Churchill, "examining the capabilities of the house.'

Now we need to prepare the inputs for the particular device (GPU or CPU) that the model is going to be run on.  Let's just check first whether GPU / CUDA has been enabled.

In [37]:
use_cuda=torch.cuda.is_available()
if use_cuda:
  print("GPU acceleration enabled")
else:
  print("GPU acceleration NOT enabled.  If using Colab, have you changed the runtype type and selected GPU as the hardware accelerator?")
device=torch.device("cuda" if use_cuda else "cpu")

GPU acceleration enabled


Lets try preparing some inputs and running them through BERT.  We will use the torch DataLoader to manage iterating over the datasets during training and testing.  Here we will just process the first item produced by the DataLoader to see the output from the pre-trained BERT model.

In [38]:
train_dataloader=torch.utils.data.DataLoader(train_data, batch_size=2, shuffle=True)

In [39]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(type(train_features))
print(list(train_features.keys()))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
['input_ids', 'token_type_ids', 'attention_mask']


In [40]:
train_features['input_ids']

tensor([[[ 101, 2273, 1997,  ...,    0,    0,    0]],

        [[ 101, 2115, 2767,  ...,    0,    0,    0]]])

In [41]:
# each individual item already has shape: [1, 512] Then DataLoader stacks two of them: [2, 1, 512]
print(train_features['input_ids'].shape) # (batch size=2 from dataloader, )
print(train_features['attention_mask'].shape)

torch.Size([2, 1, 512])
torch.Size([2, 1, 512])


[batch_size, extra_dimension, sequence_length]

`2 = batch size`
`1 = unnecessary singleton dimension`
`512 = sequence length`

that 1 appear Because EACH sample already contains a batch dimension from tokenizer.
`tokenizer(..., return_tensors='pt')` returns `[1, 512]`

Then DataLoader stacks samples:

Suppose two samples:

`sample1=  [1, 512]`
`sample2= [1, 512]`

DataLoader stacks them:

`[2, 1, 512]`


Because batch size is 2. So each batch contains:

- 2 input samples
- 2 labels

In [42]:
print(train_labels) # (sample1's label, sample2's label)
print(train_labels.shape)

tensor([1, 0])
torch.Size([2])


Squeeze is needed, because BERT expects: `[batch_size, seq_len]`

so `.squeeze(1)` we do not need second dimension

Padding `0`s are added only to make all sequences the same length, but BERT still treats them like normal tokens unless the `attention_mask` explicitly tells the model to ignore them during self-attention.

In [43]:
def prepare_inputs(input1,label,device):

  # Move labels to the target device (CPU/GPU)
  label=label.to(device)

  # Attention mask tells BERT which tokens are real (1) and which are padding (0)
  mask=input1['attention_mask'].squeeze(1).to(device)

  # Get token IDs, remove unnecessary singleton dimension, and move to device
  input_id=input1['input_ids'].squeeze(1).to(device)
  return (input_id,mask,label)

In [44]:
from transformers import BertModel

bert=BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)

# Iterate over each batch from the DataLoader
for train_input,train_label in train_dataloader:

    input_id, mask, label = prepare_inputs(train_input,train_label,device)

    # Send tokenized text through BERT
    # BERT computes: embeddings, multi-head self-attention, transformer layers and returns hidden representations.
    output = bert(input_ids=input_id, attention_mask=mask) #, return_dict=False

    break # get only one batch so I can inspect what BERT outputs look like



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
print(input_id)
print(mask)
print(label)

tensor([[ 101, 2054, 2412,  ...,    0,    0,    0],
        [ 101, 1000, 2182,  ...,    0,    0,    0]], device='cuda:0')
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')
tensor([0, 0], device='cuda:0')


In [46]:
print(len(output))
output.keys()

2


odict_keys(['last_hidden_state', 'pooler_output'])

In [47]:
output.last_hidden_state.shape
# (2     = batch size, so 2 texts/sentences
# 512   = sequence length, so 512 tokens per text
# 768   = embedding size, so each token becomes a 768-dimensional contextual vector)

torch.Size([2, 512, 768])

Now, we need to **construct our classification network.** 

Look at the code below and then answer Exercise 5

`BertModel` is ONLY the encoder. It is not yet a classifier. It produces contextual embeddings.

In [158]:
#now we need to put a simple classification layer on top of BERT

# https://docs.pytorch.org/docs/2.11/nn.html
from torch import nn # PyTorch neural network module
# https://huggingface.co/docs/transformers/en/model_doc/bert
from transformers import BertModel # pretrained BERT encoder from Hugging face transformers

# https://docs.pytorch.org/docs/2.11/generated/torch.nn.Module.html#torch.nn.Module
# This creates a custom PyTorch model.
# Because every PyTorch model inherits from:
class BertClassifier(nn.Module): 
    
    # Build the Model Architecture
    #Pretrained BERT + New classification head
    # `dropout=0.5` randomly zeroes 50% neurons during training, helps prevent overfitting
    # `num_classes=2` binary classification
    def __init__(self,dropout=0.5,num_classes=2):
        
        # Initializes nn.Module
        super(BertClassifier,self).__init__()
        
        # Load Pretrained BERT
        # This is transfer learning.
        # Instead of training language understanding from scratch:
        # we reuse pretrained linguistic knowledge
        self.bert=BertModel.from_pretrained('bert-base-uncased')
        self.dropout=nn.Dropout(dropout) # https://docs.pytorch.org/docs/2.11/generated/torch.nn.Dropout.html#torch.nn.Dropout
        self.linear=nn.Linear(768, num_classes) # https://docs.pytorch.org/docs/2.11/generated/torch.nn.Linear.html#torch.nn.Linear
        self.relu=nn.ReLU()
        # https://docs.pytorch.org/docs/2.11/nn.html#non-linear-activations-weighted-sum-nonlinearity
        # https://docs.pytorch.org/docs/2.11/generated/torch.nn.ReLU.html#torch.nn.ReLU
        
    # Forward Pass, Defines data flows through network.
    def forward(self, input_id, mask): # (token IDs from tokenizer, attention mask)
        
        # Pass Through BERT
        last_hidden_layer, pooled_output = self.bert(input_ids=input_id, attention_mask=mask, return_dict=False)
        dropout_output=self.dropout(pooled_output)
        linear_output=self.linear(dropout_output)
        final_layer=self.relu(linear_output)
        
        return final_layer

### Exercise 5

- Use the definitions of the **initialisation method** 

- and the **forward method** of 
the BertClassifier class to sketch out what the neural network architecture looks like.

What do you understand by the terms:

- pooled output

- dropout layer

- linear layer

- relu layer

##### Dropout Layer `self.dropout=nn.Dropout(dropout)`



During training: Some neurons become zero randomly.

Example:

`[0.4, 0.8, 0.1, 0.9]` -> `[0.4, 0, 0.1, 0]`

Purpose:

- reduce co-adaptation
- improve generalization
- prevent memorization

**Dropout ONLY active during:** `model.train()`
**Disabled during:** `model.eval()`


- some values became 0
- surviving values got scaled up

**Scaling**

PyTorch dropout does TWO things during training:

- Step 1 — Randomly remove neurons

`0.8  → removed → 0`

`0.5  → removed → 0`

- Step 2 — Scale surviving neurons

PyTorch divides by: $1−p$

where: `p = dropout probability`

For $p=0.5$:

$\frac{1}{1-0.5} = 2$

So surviving neurons multiply by 2.

Because expected activation magnitude should stay similar.
Otherwise activations become smaller during training.

##### Linear Classification Layer `self.linear=nn.Linear(768,num_classes)`

BERT output dimension: `768`, Because `bert-base-uncased` hidden size = 768.

So:

**Input:** 768-dimensional BERT representation
**Output:** num_classes scores

and because this is binary classification, `768 → 2`


##### ReLU Activation `self.relu=nn.ReLU()`

ReLU: `max(0,x)` , so Negative values become zero.


**Purpose:**

- introduces non-linearity into the network
- allows neural networks to learn more complex patterns

BUT:

For classification with CrossEntropyLoss,
this ReLU is usually NOT ideal.

Most modern BERT classifiers do: `return linear_output` directly.

Because: **CrossEntropyLoss expects raw logits.**


So in practice:  `final_layer=self.relu(linear_output)` is often removed.

#### Pass Through BERT

BERT returns TWO major outputs here.

##### 1) last_hidden_layer

Shape: `(batch_size, sequence_length, 768)`

Example: `torch.Size([32, 128, 768])`

Meaning: every token gets contextual embedding

##### 2) pooled_output

Shape: `(batch_size, 768)`

This is derived from the `[CLS]` token.

BERT uses: `[CLS]` as sentence representation.

So: `pooled_output = whole sentence embedding`

**This is what classification uses.**

**Why Use pooled_output?**

Because sentence classification needs: ONE vector per sentence

So:

1) `last_hidden_layer` used for: (token or entity level applications)
    - NER
    - POS tagging
    - token classification
 
2) `pooled_output` used for: (sentence level applications)
    - sentiment classification
    - spam detection
    - topic classification

##### Apply Dropout `dropout_output=self.dropout(pooled_output)`


Regularization step. Shape stays: `(batch_size,768)`

#### BERT Classifier Pipeline

```text
Input sentence
    ↓
Tokenizer
    ↓
Token IDs + Attention Mask
(batch_size, sequence_length)
    ↓
────────────────────────────
      Pretrained BERT
────────────────────────────
    ↓
Pooled Output ([CLS] representation)
last_hidden_state
(batch_size, sequence_length, 768)

pooled_output ([CLS])
(batch_size, 768)
    ↓
Dropout Layer
    ↓
Linear Classifier
(768 → num_classes)
    ↓
Class Scores / Logits
    ↓
ReLU Activation
    ↓
Final Prediction
```

**BERT provides:** contextual semantic representations

**The classifier layers learn:** how to map those representations to labels.

### Training loop


Now we will define a function which will carry out the training of the network.  

It will handle:
- setting up the DataLoaders
- preparing the inputs for CPU / GPU
- carrying out training and validation for a number of epochs.  In each epoch:
    - iterate over the training data in batches
    - get the output for each input and compute the batch loss (Cross Entropy)
    - use the batch loss to carry out optimisation
    - compute the accuracy and total loss for the training data
    - iterate over the testing / validation data in batches
    - get the output for each input and compute the batch loss
    - compute the accuracy and total loss for the validation data
    - output stats

In [60]:
#we now need a training loop
# https://docs.pytorch.org/docs/2.11/optim.html
# https://docs.pytorch.org/docs/2.11/generated/torch.optim.Adam.html#torch.optim.Adam
# https://tqdm.github.io/
from torch.optim import Adam
from tqdm import tqdm

# https://docs.pytorch.org/docs/2.11/cuda.html
# https://docs.pytorch.org/docs/2.11/meta.html device

def train(model, train_data, val_data, learning_rate, epochs):
    
    # Create data loaders
    # shuffle=True Prevents model learning sequence/order patterns.
    train_dataloader=torch.utils.data.DataLoader(train_data, batch_size=2, shuffle=True)
    # shuffle=False to keep evaluation set consistent
    val_dataloader=torch.utils.data.DataLoader(val_data, batch_size=2)
    
    # Tensors moves to cuda
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    # https://docs.pytorch.org/docs/2.11/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss
    # Loss Function
    criterion=nn.CrossEntropyLoss()
    # Optimizer
    optimizer=Adam(model.parameters(), lr=learning_rate)
    
    if use_cuda:
        model=model.cuda()
        criterion=criterion.cuda()


    # Epoch Loop    
    for epoch_num in range(epochs):

        # Metrics Initialization
        total_acc_train = 0 # cumulative accuracy
        total_loss_train = 0 # cumulative loss

        # Training Mode
        #Activates: dropout and batchnorm updates
        model.train()

        # Batch Loop Iterates batch-by-batch.
        for train_input,train_label in tqdm(train_dataloader):
            
            # Prepare Inputs
            input_id, mask, train_label = prepare_inputs(train_input, train_label, device)
            
            # Forward Pass
            output = model(input_id, mask)
            
            # Compute Loss
            # .long() converts labels to integer typ
            batch_loss = criterion(output, train_label.long())

            # Store Loss 
            # .item()extracts Python scalar from tensor.
            total_loss_train += batch_loss.item() 
            
            # Compute Accuracy
            # output.argmax(dim=1) Largest logit index chosen.
            # Compare with labels
            # Count correct predictions
            acc = (output.argmax(dim=1) == train_label).sum().item()
            total_acc_train += acc 
            
            # Zero Gradients PyTorch accumulates gradients by default.
            # Without this: gradients stack endlessly.
            model.zero_grad()

            # Backpropagation, Computes gradients using chain rule.
            # This is: automatic differentiation, autograd
            batch_loss.backward()

            # Optimizer Step, Updates weights.
            #
            optimizer.step()

            
        total_acc_val=0
        total_loss_val=0
        predictions = []
        true_labels = []

        # Validation Phase
        model.eval()

        # Disable Gradient Computation
        with torch.no_grad():

            for val_input,val_label in val_dataloader:
                
                input_id, mask, val_label = prepare_inputs(val_input, val_label, device)
                
                output = model(input_id, mask)
                
                batch_loss = criterion(output, val_label.long())
                
                total_loss_val += batch_loss.item()
                
                preds = output.argmax(dim=1)

                predictions.extend(preds.cpu().numpy())
                true_labels.extend(val_label.cpu().numpy())

                acc = (preds == val_label).sum().item()
                total_acc_val += acc
                
        print(f'Epochs: {epoch_num+1} | Train Loss: {total_loss_train / len(train_data):.3f} | Train Accuracy: {total_acc_train / len(train_data):.3f}')
        print(f'Val loss: {total_loss_val / len(val_data):.3f} | Val Accuracy: {total_acc_val / len(val_data):.3f}')

    
    return predictions, true_labels

Here, we define the number of epochs we are going to train for, the learning rate and an instance of our BertClassifier network.

In [61]:
EPOCHS=1  
LR=1e-6
model=BertClassifier(num_classes=len(labels.keys()))

#this will freeze the pre-trained BERT model and just make the classification head trainable
#can speed things up and avoid "catastrophic forgetting" / overfitting on task-specific data
model.bert.requires_grad_(False) 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Now we are actually going to train the model.  This might take some time - particularly if you are running on a CPU (1.5 hrs per epoch on my laptop!).  I recommend running on colab with GPU enabled (or using the lab machines with GPU).

In [53]:
print(next(model.parameters()).device)

cpu


In [62]:
predictions_valid, true_labels_valid = train(model, train_data, valid_data, LR, EPOCHS)

100%|██████████| 1508/1508 [00:30<00:00, 48.82it/s]


Epochs: 1 | Train Loss: 0.360 | Train Accuracy: 0.495
Val loss: 0.352 | Val Accuracy: 0.495


We need to be able to save the model to be able to use it elsewhere (without training again!)

In [159]:

output_dir="bert-base-uncased-bookclassifier"
torch.save(model,output_dir)

PicklingError: Can't pickle <class '__main__.BertClassifier'>: it's not the same object as __main__.BertClassifier

We can load it up like this.  This could be in another notebook.  If loading in another notebook, you should make sure the BertClassifier class is also defined in that notebook (along with other necessary imports).

In [ ]:

#input_dir="Lab9resources/julie-bert-base-uncased-bookclassifier"
model = torch.load("bert-base-uncased-bookclassifier",
                    weights_only=False,
                    map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")
                    )

Here's an evaluation loop we can use to evaluate on some test data.  We also return the predictions which can than be added to the dataframe

In [94]:
def evaluate(model,test_dataset, batch_size=2):
    model.eval()
    test_dataloader=torch.utils.data.DataLoader(test_dataset, batch_size)
    
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    if use_cuda:
        model=model.cuda()
        
    total_acc_test=0

    with torch.no_grad():
    
        count=0
        predictions=[]
    
        for test_input,test_label in tqdm(test_dataloader):
    
            count += batch_size
    
            test_label=test_label.to(device).long()
    
            mask=test_input['attention_mask'].squeeze(1).to(device)
            input_id=test_input['input_ids'].squeeze(1).to(device)
    
            output=model(input_id,mask)
    
            #print(output.argmax(dim=1),test_label)
            predictions.append(output.argmax(dim=1))  #save the prediction for further analysis
            acc=(output.argmax(dim=1)==test_label).sum().item()
            
            total_acc_test+=acc
            
            if count%100==0:
                print(f'Accuracy so far = {total_acc_test/count: .3f}')
            
    print(f'Test accuracy: {total_acc_test/len(test_dataset): .3f}')
    return predictions

This takes around 6 minutes to run on my laptop.  I have the evaluation loop printing out "Accuracy so far" as I get bored waiting to the end to see the results - it also gives a very rough indication of how stable the method is

In [95]:
predictions=evaluate(model, test_data)

 32%|███▏      | 60/189 [00:01<00:02, 52.69it/s]

Accuracy so far =  0.030


 57%|█████▋    | 108/189 [00:02<00:01, 51.71it/s]

Accuracy so far =  0.025


 83%|████████▎ | 156/189 [00:03<00:00, 50.04it/s]

Accuracy so far =  0.337


100%|██████████| 189/189 [00:03<00:00, 51.62it/s]

Test accuracy:  0.469


In [ ]:
predictions

[tensor([1, 0], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 0], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1], device='cuda:0'),
 tensor([1, 1]

### Exercise 6
Add the predicted label for each test item to the dataframe with the test data.

In [108]:
predictions_numpy = []

for batch_preds in predictions:
    predictions_numpy.extend(batch_preds.cpu().numpy())

In [109]:
reverse_index

{0: 'Emma', 1: 'Ivanhoe'}

In [113]:
named_predictions = []
for batch in predictions:
    for pred in batch.tolist():  # .tolist() converts tensor to plain python list
        named_predictions.append(reverse_index[pred])

named_predictions

['Ivanhoe',
 'Emma',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Emma',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe',
 'Ivanhoe'

In [129]:
testing_df["predicted_label"] =named_predictions

In [130]:
testing_df

,text,label,predicted_label
0,"I think him a very handsome young man, and his...",Emma,Ivanhoe
1,Mr. Knightley and Harriet!--It was an odd tete...,Emma,Emma
2,"Captain Weston, who had been considered, espec...",Emma,Ivanhoe
3,I hope I have a better taste than to think of ...,Emma,Ivanhoe
4,She laughed because she was disappointed; and ...,Emma,Ivanhoe
...,...,...,...
372,"And there was such a solemn melody, 'Twixt dol...",Ivanhoe,Ivanhoe
373,"``The castle burns,'' said Rebecca; ``it burns...",Ivanhoe,Ivanhoe
374,"and for the evil which thou hast sustained, se...",Ivanhoe,Ivanhoe
375,But although the lists rang with the applauses...,Ivanhoe,Ivanhoe


In [135]:
filter = testing_df["predicted_label"] == testing_df["label"]
testing_df.loc[filter].count()

text               177
label              177
predicted_label    177
dtype: int64

In [138]:
testing_df["predicted_label"].value_counts()

predicted_label
Ivanhoe    370
Emma         7
Name: count, dtype: int64

In [ ]:
from collections import Counter
print("Test Predictions:", Counter(predictions_numpy))


Test Predictions: Counter({np.int64(1): 370, np.int64(0): 7})


In [97]:
test_dataloader=torch.utils.data.DataLoader(test_data, batch_size=2)
all_test_labels = []

for test_input, test_label in test_dataloader:
    all_test_labels.extend(test_label.cpu().numpy())

print(Counter(all_test_labels))

Counter({np.int64(0): 203, np.int64(1): 174})


The Model predict everything `0` `Emma` class. This is suspicious.

In [ ]:
print("Train labels:", Counter(train_labels))
print("Val labels:", Counter(true_labels_valid))
print("Predictions:", Counter(predictions_valid))

Train labels: Counter({tensor(1): 1, tensor(0): 1})
Val labels: Counter({np.int64(0): 203, np.int64(1): 175})
Predictions: Counter({np.int64(1): 366, np.int64(0): 12})


In [ ]:
train_dataloader=torch.utils.data.DataLoader(train_data, batch_size=2, shuffle=True)
all_train_labels = []

for train_input, train_label in train_dataloader:
    all_train_labels.extend(train_label.cpu().numpy())

print(Counter(all_train_labels))

Counter({np.int64(0): 1622, np.int64(1): 1394})


### Essential BugFix

After seeing validation and test `accuracy`, initially I suspected that the model make predictions just about the `random`. Even classes fairly balance approx. `50% accuracy` was a clue the model did not train well.

However, once inspecting `the predictions`, I noticed that the model predicted almost everything as class 1.

But, the problem is not train labels imbalance. Because:
- Class Emma = 1622 count
- Class Ivanhoe = 1394 count

**MAIN PROBLEM**

`final_layer = self.relu(linear_output)` that.

Because `CrossEntropyLoss` expects raw logits, but `ReLU` changes negative values to zero. This can damage the class competition and make the model collapse toward one class.

Thus, I removed `ReLU`, `BertClassifier` classifier will return logits now.

In [168]:
#now we need to put a simple classification layer on top of BERT

# https://docs.pytorch.org/docs/2.11/nn.html
from torch import nn # PyTorch neural network module
# https://huggingface.co/docs/transformers/en/model_doc/bert
from transformers import BertModel # pretrained BERT encoder from Hugging face transformers

# https://docs.pytorch.org/docs/2.11/generated/torch.nn.Module.html#torch.nn.Module
# This creates a custom PyTorch model.
# Because every PyTorch model inherits from:
class BertClassifier(nn.Module): 
    
    # Build the Model Architecture
    #Pretrained BERT + New classification head
    # `dropout=0.5` randomly zeroes 50% neurons during training, helps prevent overfitting
    # `num_classes=2` binary classification
    def __init__(self,dropout=0.5,num_classes=2):
        
        # Initializes nn.Module
        super(BertClassifier,self).__init__()
        
        # Load Pretrained BERT
        # This is transfer learning.
        # Instead of training language understanding from scratch:
        # we reuse pretrained linguistic knowledge
        self.bert=BertModel.from_pretrained('bert-base-uncased')
        self.dropout=nn.Dropout(dropout) # https://docs.pytorch.org/docs/2.11/generated/torch.nn.Dropout.html#torch.nn.Dropout
        self.linear=nn.Linear(768, num_classes) # https://docs.pytorch.org/docs/2.11/generated/torch.nn.Linear.html#torch.nn.Linear

        # https://docs.pytorch.org/docs/2.11/nn.html#non-linear-activations-weighted-sum-nonlinearity
        # https://docs.pytorch.org/docs/2.11/generated/torch.nn.ReLU.html#torch.nn.ReLU
        
    # Forward Pass, Defines data flows through network.
    def forward(self, input_id, mask): # (token IDs from tokenizer, attention mask)
        
        # Pass Through BERT
        last_hidden_layer, pooled_output = self.bert(input_ids=input_id, attention_mask=mask, return_dict=False)
        dropout_output=self.dropout(pooled_output)
        logits=self.linear(dropout_output)
  
        
        return logits

In [139]:
EPOCHS=1  
LR=1e-6
second_model=BertClassifier(num_classes=len(labels.keys()))

#this will freeze the pre-trained BERT model and just make the classification head trainable
#can speed things up and avoid "catastrophic forgetting" / overfitting on task-specific data
second_model.bert.requires_grad_(False) 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [140]:
second_predictions_valid, second_true_labels_valid = train(second_model, train_data, valid_data, LR, EPOCHS)

100%|██████████| 1508/1508 [00:31<00:00, 48.59it/s]


Epochs: 1 | Train Loss: 0.356 | Train Accuracy: 0.523
Val loss: 0.339 | Val Accuracy: 0.590


In [142]:
print("Train labels:", Counter(all_train_labels))
print("Val labels:", Counter(second_true_labels_valid))
print("Predictions:", Counter(second_predictions_valid))

Train labels: Counter({np.int64(0): 1622, np.int64(1): 1394})
Val labels: Counter({np.int64(0): 203, np.int64(1): 175})
Predictions: Counter({np.int64(0): 320, np.int64(1): 58})


The classifier is learning a little, but still underfitting and class-biased.

Most likely reasons:

1. Frozen $BERT + LR=1e-6$ is too slow
2. Dropout=0.5 is too strong
3. Only 1 epoch is not enough

After evaluation I will train again and compare them.

### Exercise 7
Compute the confusion matrix / precision and recall scores for the different classes.  What does this analysis tell you about the errors?

In [143]:
secoond_test_predictions = evaluate(second_model, test_data)

 29%|██▉       | 55/189 [00:01<00:02, 51.05it/s]

Accuracy so far =  0.910


 58%|█████▊    | 109/189 [00:02<00:01, 50.32it/s]

Accuracy so far =  0.920


 83%|████████▎ | 157/189 [00:03<00:00, 51.41it/s]

Accuracy so far =  0.703


100%|██████████| 189/189 [00:03<00:00, 49.41it/s]

Test accuracy:  0.599


In [156]:
torch.save(second_model.state_dict(), "second_bookclassifier_weights.pt")

In [151]:
true_labels_test = testing_df["label"].tolist()
true_labels_test[:5]

['Emma', 'Emma', 'Emma', 'Emma', 'Emma']

In [150]:
secoond_test_predictions_labels = []
for batch in secoond_test_predictions:
    for i in batch.tolist():
        secoond_test_predictions_labels.append(reverse_index[i])
secoond_test_predictions_labels[:5]

['Emma', 'Emma', 'Emma', 'Emma', 'Emma']

In [152]:
from sklearn.metrics import classification_report

print(classification_report(true_labels_test, secoond_test_predictions_labels))

              precision    recall  f1-score   support

        Emma       0.58      0.92      0.71       203
     Ivanhoe       0.71      0.22      0.34       174

    accuracy                           0.60       377
   macro avg       0.64      0.57      0.53       377
weighted avg       0.64      0.60      0.54       377



In [153]:
third_model = BertClassifier(dropout=0.1, num_classes=2)
LR_2 = 2e-5
EPOCHS_2 = 3
third_model.bert.requires_grad_(False) 


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [154]:
third_predictions_valid, third_true_labels_valid = train(third_model, train_data, valid_data, LR_2, EPOCHS_2)

100%|██████████| 1508/1508 [00:31<00:00, 47.37it/s]


Epochs: 1 | Train Loss: 0.321 | Train Accuracy: 0.662
Val loss: 0.294 | Val Accuracy: 0.749


100%|██████████| 1508/1508 [00:32<00:00, 46.89it/s]


Epochs: 2 | Train Loss: 0.288 | Train Accuracy: 0.752
Val loss: 0.263 | Val Accuracy: 0.820


100%|██████████| 1508/1508 [00:32<00:00, 45.87it/s]


Epochs: 3 | Train Loss: 0.266 | Train Accuracy: 0.791
Val loss: 0.244 | Val Accuracy: 0.852


In [155]:
torch.save(third_model.state_dict(), "third_bookclassifier_weights.pt")

In [161]:
third_test_predictions = evaluate(third_model, test_data)

 30%|██▉       | 56/189 [00:01<00:02, 50.73it/s]

Accuracy so far =  0.740


 58%|█████▊    | 110/189 [00:02<00:01, 51.98it/s]

Accuracy so far =  0.760


 84%|████████▎ | 158/189 [00:03<00:00, 51.56it/s]

Accuracy so far =  0.797


100%|██████████| 189/189 [00:03<00:00, 48.73it/s]

Test accuracy:  0.814


In [162]:
true_labels_test = testing_df["label"].tolist()
true_labels_test[:5]

['Emma', 'Emma', 'Emma', 'Emma', 'Emma']

In [165]:
third_test_predictions_labels = []
for batch in third_test_predictions:
    for i in batch.tolist():
        third_test_predictions_labels.append(reverse_index[i])
third_test_predictions_labels[:5]

['Emma', 'Emma', 'Ivanhoe', 'Emma', 'Emma']

In [166]:
from sklearn.metrics import classification_report

print(classification_report(true_labels_test, third_test_predictions_labels))

              precision    recall  f1-score   support

        Emma       0.88      0.76      0.82       203
     Ivanhoe       0.76      0.87      0.81       174

    accuracy                           0.81       377
   macro avg       0.82      0.82      0.81       377
weighted avg       0.82      0.81      0.81       377



In [170]:
full_tuning_model = BertClassifier(dropout=0.1, num_classes=2)
LR_2 = 2e-5
EPOCHS_2 = 3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [171]:
full_tuning_predictions_valid, true_labels_valid = train(full_tuning_model, train_data, valid_data, LR_2, EPOCHS_2)

100%|██████████| 1508/1508 [01:35<00:00, 15.85it/s]


Epochs: 1 | Train Loss: 0.049 | Train Accuracy: 0.963
Val loss: 0.016 | Val Accuracy: 0.992


100%|██████████| 1508/1508 [01:35<00:00, 15.79it/s]


Epochs: 2 | Train Loss: 0.014 | Train Accuracy: 0.993
Val loss: 0.010 | Val Accuracy: 0.992


100%|██████████| 1508/1508 [01:35<00:00, 15.85it/s]


Epochs: 3 | Train Loss: 0.007 | Train Accuracy: 0.995
Val loss: 0.013 | Val Accuracy: 0.989


In [172]:
torch.save(full_tuning_model.state_dict(), "fulltune_bookclassifier_weights.pt")